# Fine-tuning Médical — Phi-3.5-mini avec QLoRA 4-bit

> **Mission Expérimentale — TechCorp Hackathon**  
> Stack : HuggingFace Transformers · PEFT/LoRA · BitsAndBytes · TRL SFTTrainer  
> Dataset : [ruslanmv/ai-medical-chatbot](https://huggingface.co/datasets/ruslanmv/ai-medical-chatbot)  
> Durée estimée : ~35–50 min sur GPU T4

---
⚠️ **Disclaimer** : Modèle expérimental — ne pas utiliser pour des décisions médicales réelles.

In [1]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU non détecté. Activez-le : Runtime > Change runtime type > T4 GPU')

gpu = torch.cuda.get_device_properties(0)
print(f'GPU         : {gpu.name}')
print(f'VRAM totale : {gpu.total_memory / 1e9:.1f} GB')
print(f'CUDA        : {torch.version.cuda}')

GPU         : Tesla T4
VRAM totale : 15.6 GB
CUDA        : 12.8


In [2]:
# transformers>=4.47.0 requis — processing_class ajouté dans 4.47 (incompatible avec 4.45)
!pip install -q 'transformers>=4.47.0' 'bitsandbytes>=0.43.0' 'peft>=0.12.0' 'trl>=0.12.0' 'accelerate>=0.34.0' 'datasets>=2.20.0'
print('Installation terminée.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.3 MB/s eta 0:00:00
Installation terminée.


In [3]:
MODEL_NAME    = 'microsoft/Phi-3.5-mini-instruct'
DATASET_NAME  = 'ruslanmv/ai-medical-chatbot'
NUM_EXAMPLES  = 2000
MAX_SEQ_LEN   = 1024
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05
BATCH_SIZE    = 2
GRAD_ACCUM    = 4
LEARNING_RATE = 2e-4
NUM_EPOCHS    = 1
OUTPUT_DIR    = './medical_lora_adapter'

SYSTEM_PROMPT = (
    'You are a knowledgeable medical assistant. '
    'Provide accurate, evidence-based medical information. '
    'Always recommend consulting a qualified healthcare professional '
    'for diagnosis and treatment decisions.'
)

print(f'Modèle  : {MODEL_NAME}')
print(f'Dataset : {DATASET_NAME} ({NUM_EXAMPLES} exemples)')
print(f'LoRA    : r={LORA_R}, alpha={LORA_ALPHA}')

Modèle  : microsoft/Phi-3.5-mini-instruct
Dataset : ruslanmv/ai-medical-chatbot (2000 exemples)
LoRA    : r=16, alpha=32


## 1. Dataset

In [4]:
from datasets import load_dataset

print(f'Chargement : {DATASET_NAME}...')
raw = load_dataset(DATASET_NAME, split='train')
print(f'Exemples totaux : {len(raw):,}')
print(f'Colonnes        : {raw.column_names}')

row = raw[0]
q = row.get('Patient', row.get('input', row.get('question', '')))
a = row.get('Doctor',  row.get('output', row.get('answer',  '')))
print(f'\nExemple Patient : {str(q)[:150]}')
print(f'Exemple Doctor  : {str(a)[:150]}')

Chargement : ruslanmv/ai-medical-chatbot...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/863 [00:00<?, ?B/s]

dialogues.parquet:   0%|          | 0.00/142M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/256916 [00:00<?, ? examples/s]

Exemples totaux : 256,916
Colonnes        : ['Description', 'Patient', 'Doctor']

Exemple Patient : Hi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for an
Exemple Doctor  : Hi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist


In [5]:
def format_example(example):
    q = example.get('Patient', example.get('input',  example.get('question', '')))
    a = example.get('Doctor',  example.get('output', example.get('answer',   '')))
    if not q or not a:
        return {'text': ''}
    text = (
        f'<|system|>\n{SYSTEM_PROMPT}<|end|>\n'
        f'<|user|>\n{str(q).strip()}<|end|>\n'
        f'<|assistant|>\n{str(a).strip()}<|end|>'
    )
    return {'text': text}

subset  = raw.select(range(min(NUM_EXAMPLES, len(raw))))
dataset = subset.map(format_example, num_proc=2)
dataset = dataset.filter(lambda x: len(x['text']) > 100)
dataset = dataset.remove_columns([c for c in dataset.column_names if c != 'text'])

split   = dataset.train_test_split(test_size=0.05, seed=42)
train_d = split['train']
val_d   = split['test']

print(f'Train : {len(train_d):,} exemples')
print(f'Val   : {len(val_d):,} exemples')
print(f"\nExemple formaté :\n{train_d[0]['text'][:350]}...")

Map (num_proc=2):   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

Train : 1,900 exemples
Val   : 100 exemples

Exemple formaté :
<|system|>
You are a knowledgeable medical assistant. Provide accurate, evidence-based medical information. Always recommend consulting a qualified healthcare professional for diagnosis and treatment decisions.<|end|>
<|user|>
Hello doctor, My boyfriend is 24 years old. Being a fitness trainer, he does not take any drugs. He has been getting chest ...


## 2. Modèle + LoRA

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

print('Chargement du tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

print(f'Chargement de {MODEL_NAME} en 4-bit...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    attn_implementation='eager',
)
model = prepare_model_for_kbit_training(model)

total = sum(p.numel() for p in model.parameters())
print(f'Modèle chargé — {total/1e9:.2f}B paramètres')

Chargement du tokenizer...


config.json:   0%|          | 0.00/3.45k [00:00<?, ?B/s]

configuration_phi3.py:   0%|          | 0.00/11.2k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3.5-mini-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json:   0%|          | 0.00/3.98k [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

Modèle chargé — 2.01B paramètres


In [7]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Paramètres entraînables : {trainable:,} ({100*trainable/total:.2f}%)')

Paramètres entraînables : 8,912,896 (0.44%)


## 3. Entraînement

In [ ]:
# TRL >= 0.14 : SFTConfig regroupe tout ; tokenizer -> processing_class ; max_seq_length -> max_length
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    # --- SFT-specific ---
    dataset_text_field          = 'text',
    max_length                  = MAX_SEQ_LEN,
    packing                     = False,
    # --- Training ---
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    warmup_ratio                = 0.05,
    lr_scheduler_type           = 'cosine',
    logging_steps               = 25,
    save_steps                  = 200,
    save_total_limit            = 2,
    eval_strategy               = 'steps',
    eval_steps                  = 100,
    bf16                        = torch.cuda.is_bf16_supported(),
    fp16                        = not torch.cuda.is_bf16_supported(),
    optim                       = 'paged_adamw_8bit',
    weight_decay                = 0.01,
    gradient_checkpointing      = True,
    gradient_checkpointing_kwargs = {'use_reentrant': False},
    report_to                   = 'none',
    seed                        = 42,
)

trainer = SFTTrainer(
    model           = model,
    processing_class = tokenizer,   # 'tokenizer' renommé 'processing_class' dans trl récent
    train_dataset   = train_d,
    eval_dataset    = val_d,
    args            = sft_config,
)

print('Démarrage de l\'entraînement...')
stats = trainer.train()
print(f'\nTerminé !')
print(f'  Loss  : {stats.training_loss:.4f}')
print(f'  Durée : {stats.metrics["train_runtime"]:.0f}s')

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/1900 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1900 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1900 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Démarrage de l'entraînement...


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is de

Step,Training Loss,Validation Loss


## 4. Test du modèle

In [ ]:
model.eval()

def generate(question, max_new_tokens=256):
    prompt = (
        f'<|system|>\n{SYSTEM_PROMPT}<|end|>\n'
        f'<|user|>\n{question}<|end|>\n'
        f'<|assistant|>\n'
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).replace('<|end|>', '').strip()

questions = [
    'What are the main symptoms of type 2 diabetes?',
    'How does hypertension affect the cardiovascular system?',
    'What is the difference between viral and bacterial pneumonia?',
    'Explain the mechanism of action of beta-blockers.',
    'What are the first-line treatments for major depressive disorder?',
]

print('=' * 65)
for q in questions:
    print(f'\nQ : {q}')
    print(f'R : {generate(q)}')
    print('-' * 65)

## 5. Sauvegarde et téléchargement

In [ ]:
import os, shutil
from google.colab import files

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

saved = [(f, os.path.getsize(os.path.join(OUTPUT_DIR, f))/1e6)
         for f in os.listdir(OUTPUT_DIR)]
print('Fichiers sauvegardés :')
for name, size in sorted(saved):
    print(f'  {name:<45} {size:.1f} MB')

shutil.make_archive('medical_lora_adapter', 'zip', OUTPUT_DIR)
zip_size = os.path.getsize('medical_lora_adapter.zip') / 1e6
print(f'\nArchive : medical_lora_adapter.zip ({zip_size:.0f} MB)')
files.download('medical_lora_adapter.zip')